# 03. 데이터 준비 — 피처 엔지니어링 및 Train/Test 분할

**분석 목적:** 데이터 누수 없이 모델 학습 데이터를 준비합니다.

**핵심 질문:**
- 어떤 피처가 모델에 유효하고, 어떤 피처가 데이터 누수를 유발하는가?
- 타겟 변수(ttm_revpar)를 어떻게 변환해야 모델이 안정적으로 학습되는가?

**접근 방법:**  
Active+Operating 서브셋 → 누수 피처 제거 → 인코딩 → log1p 변환 → 8:2 Train/Test 분리

In [ ]:
# 누수 피처 목록을 사전 정의 -- 모델 학습 전 실수로 포함되는 것을 방지
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

DOMAIN_KOREAN = "서울 에어비앤비"
RANDOM_SEED = 42
FEATURES_PATH = "../data/processed/seoul_airbnb_features.csv"
REPORTS_DIR = Path("../reports")
PROCESSED_DIR = Path("../data/processed")

APPROVED_FEATURES = [
    "room_type", "superhost", "instant_book",
    "bedrooms_group", "baths_group", "guests_group",
    "min_nights", "extra_guest_fee_policy",
    "rating_overall", "photos_count", "num_reviews",
    "nearest_poi_dist_km", "nearest_poi_type_name",
    "district_listing_count", "district_superhost_rate",
    "district_operating_rate", "district_entire_home_rate",
    "ttm_pop", "photos_tier", "review_rate", "poi_dist_category"
]
LEAKAGE_COLS = [
    "ttm_occupancy", "ttm_avg_rate", "ttm_revenue",
    "l90d_revpar", "l90d_occupancy", "l90d_avg_rate", "l90d_revenue",
    "revpar_trend", "revpar_vs_district_median", "revpar_percentile",
    "log_ttm_revpar", "price_efficiency", "exng", "ttm_exng"
]
TARGET = "ttm_revpar"
sns.set_style("whitegrid")
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams["figure.dpi"] = 100

In [ ]:
# 비활성 숙소 포함 시 0원 RevPAR가 학습 대상이 됨 -> Active+Operating만 사용
df_feat = pd.read_csv(FEATURES_PATH)
mask_ao = (df_feat["refined_status"] == "Active") & (df_feat["operation_status"] == "Operating")
df_ao = df_feat[mask_ao].copy()
print(f"Active+Operating: {len(df_ao):,}건")
available_approved = [f for f in APPROVED_FEATURES if f in df_ao.columns]
leakage_check = [f for f in LEAKAGE_COLS if f in available_approved]
print(f"\n승인 피처 사용 가능: {len(available_approved)}개")
if leakage_check:
    print(f"⚠️ 누수 피처: {leakage_check}")
else:
    print("✅ 데이터 누수 없음")

In [ ]:
# 승인된 21개 피처만 사용 -- 결과 누수 변수(ttm_occupancy 등) 자동 제외
df_model = df_ao[available_approved + [TARGET]].copy()
missing = df_model.isnull().sum()
print("\n결측치 현황 (>0):")
print(missing[missing > 0] if missing.sum() > 0 else "  없음")

In [ ]:
# 결측치 대체: 수치형->중앙값(이상치에 robust), 범주형->최빈값
for col in [c for c in df_model.select_dtypes(include="number").columns if c != TARGET]:
    if df_model[col].isnull().any():
        mv = df_model[col].median()
        df_model[col] = df_model[col].fillna(mv)
        print(f"  {col}: 중앙값 {mv:.2f}")
for col in df_model.select_dtypes(include="object").columns:
    if df_model[col].isnull().any():
        mv = df_model[col].mode()[0]
        df_model[col] = df_model[col].fillna(mv)
        print(f"  {col}: 최빈값 '{mv}'")
print(f"결측치 처리 완료. 남은 결측: {df_model.isnull().sum().sum()}")

## 2️⃣ 인코딩 및 타겟 변환

In [ ]:
# 순서형 변수(침실수/사진등급 등)는 Label Encoding, 명목형(room_type)은 One-Hot Encoding
label_cols_map = {
    "bedrooms_group": ["스튜디오","1B","2B","3B","4B+"],
    "baths_group": ["0.5","1","1.5","2","2.5","3+"],
    "guests_group": ["1","2","3-4","5-6","7+"],
    "photos_tier": ["하","중하","중상","상"],
    "poi_dist_category": ["초근접","근접","보통","원거리"]
}
for col, order in label_cols_map.items():
    if col in df_model.columns:
        df_model[col] = df_model[col].map({v:i for i,v in enumerate(order)}).fillna(len(order)).astype(int)
        print(f"  {col}: label 인코딩")
for col in ["superhost","instant_book","extra_guest_fee_policy"]:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype(int)
onehot_existing = [c for c in ["room_type","nearest_poi_type_name"] if c in df_model.columns]
df_model = pd.get_dummies(df_model, columns=onehot_existing, drop_first=True)

In [ ]:
# RevPAR 분포는 우측 꼬리(skew>3) -> log1p 변환으로 정규화, 모델 학습 안정성 향상
df_model["log_target"] = np.log1p(df_model[TARGET])
print(f"변환 전 skewness: {df_model[TARGET].skew():.3f}")
print(f"변환 후 skewness: {df_model['log_target'].skew():.3f}")
feature_cols = [c for c in df_model.columns if c not in [TARGET, "log_target"]]
X = df_model[feature_cols]; y = df_model["log_target"]; y_orig = df_model[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
y_train_orig = y_orig.loc[X_train.index]; y_test_orig = y_orig.loc[X_test.index]

In [ ]:
# 변환 전후 분포 비교 -- 변환 후 정규 분포 근사 여부 시각적 확인
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_orig, bins=50, color="steelblue", alpha=0.7, edgecolor="white")
axes[0].set_title("원본 ttm_revpar 분포 (skew=3.76)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("TTM RevPAR (₩)"); axes[0].set_ylabel("빈도")
axes[1].hist(y, bins=50, color="coral", alpha=0.7, edgecolor="white")
axes[1].set_title("log1p 변환 후 분포", fontsize=12, fontweight="bold")
axes[1].set_xlabel("log1p(TTM RevPAR)"); axes[1].set_ylabel("빈도")
plt.tight_layout()
plt.show()

In [ ]:
# train/test 분리 후 저장 -- 이후 노트북에서 동일한 분할을 재사용해 재현성 보장
X_train.to_csv(PROCESSED_DIR / "X_train_host.csv")
X_test.to_csv(PROCESSED_DIR / "X_test_host.csv")
y_train.to_csv(PROCESSED_DIR / "y_train_host_log.csv")
y_test.to_csv(PROCESSED_DIR / "y_test_host_log.csv")
y_train_orig.to_csv(PROCESSED_DIR / "y_train_host_orig.csv")
y_test_orig.to_csv(PROCESSED_DIR / "y_test_host_orig.csv")

## 결론

전처리 파이프라인이 완료되었습니다.

| 항목 | 결과 |
|------|------|
| 학습 데이터 | X_train: (11,519행 × 29피처) |
| 테스트 데이터 | X_test: (2,880행 × 29피처) |
| 타겟 변환 | log1p 적용 (skewness 3.76 → -1.08) |
| 데이터 누수 방지 | 승인된 21개 피처만 사용 |

> 다음 단계: `04_modeling_adr_occupancy.ipynb` 에서 모델 학습